In [13]:
import math
import random
import pandas as pd
import re
from io import StringIO

def get_instance(instance: str, vtype: int, folder: str, name: str):
    # Nodes
    N = []
    N_S = []
    N_C = []
    N_R = []

    # Customer parameters
    q = {}
    S = {}
    e = {}
    l = {}
    p = {}

    # Vehicles
    T = []
    F = []

    # Vehicle parameters
    Q = {}
    L = {}
    h = {}
    c = {}

    # Read instance
    trucks_parsed = False
    freighters_parsed = False
    satellites_parsed = False
    customers_parsed = False

    V = {}
    coord = {}
    if vtype == 1:
        hv = [1.0]
    elif vtype == 2:
        hv = [0.9, 1.1]
    elif vtype == 3:
        hv = [0.8, 1.0, 1.2]

    with open(instance) as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("!"):
                continue

            # First-echelon trucks
            if not trucks_parsed:
                tn_s, tQ_s, tc_s, th_s = line.split(",")
                tn, tQ, tc, th = int(tn_s), int(tQ_s), int(tc_s), int(th_s)

                t = []
                Tn = tn

                for i, vt in enumerate(hv):
                    htn = round(Tn/len(hv[i:]))
                    t.append([int(htn), int(round((vt**2)*tQ)), int(tc), int(round(vt*(th if th > 0 else 150*tc)))])
                    Tn -= htn

                T = t   
                trucks_parsed = True
                continue

            # Second-echelon electric vehicles
            if not freighters_parsed:
                fns_s, fn_s, fQ_s, fc_s, fh_s, fL_s, frho_s = line.split(",")
                fns, fn, fQ, fc, fh, fL, frho = int(fns_s), int(fn_s), int(fQ_s), int(fc_s), int(fh_s), int(fL_s), int(frho_s)
                
                t = []
                Fn = fn
                Fns = fns

                for i, vt in enumerate(hv):
                    hfns = round(Fns/len(hv[i:]))
                    hfn = round(Fn/len(hv[i:]))
                    t.append([int(hfns), int(hfn), int(round((vt**2)*fQ)), int(fc), int(round(vt*(fh if th > 0 else 80*fc))), int(round((vt**2)*fL)), int(frho)])
                    Fns -= hfns
                    Fn -= hfn

                F = t
                freighters_parsed = True
                continue

            # Satellites
            if not satellites_parsed:
                sat_tokens = line.split()

                for idx, token in enumerate(sat_tokens):
                    if idx == 0:
                        # Deposit
                        N_d = "D0"
                        x0, y0 = token.split(",")
                        coord[N_d] = (float(x0), float(y0))
                        N.append(N_d)                       
                    else:
                        # Satellite
                        s_id = f"S{idx-1}"
                        xs, ys, _, _, _ = token.split(",")
                        coord[s_id] = (float(xs), float(ys))
                        N.append(s_id)
                        N_S.append(s_id)

                satellites_parsed = True

                continue

            # Customers
            if not customers_parsed:
                cust_tokens = line.split()
                for idx, token in enumerate(cust_tokens):
                    c_id = f"C{idx}"
                    xc, yc, q_s = token.split(",")
                    coord[c_id] = (float(xc), float(yc))
                    N.append(c_id)
                    N_C.append(c_id)
                    q[c_id] = float(q_s)

                customers_parsed = True
                continue

            # Recharging stations
            stations_tokens = line.split()
            for idx, token in enumerate(stations_tokens):
                r_id = f"R{idx}"
                xr, yr = token.split(",")
                coord[r_id] = (float(xr), float(yr))
                N.append(r_id)
                N_R.append(r_id)

        # Distance
        t = {}
        for i in N:
            xi, yi = coord[i]
            for j in N:
                xj, yj = coord[j]
                dij = round(math.sqrt((xi - xj)**2 + (yi - yj)**2))
                t[i, j] = dij

        #Service time
        tt_ = 0
        total_q = sum(q.values())
        n_sat = len(N_S)
        n_cust = len(N_C)

        for i in N_C:
            for s in N_S:
                tt_ += t[s, i] + t[i, s]
        
        tt_ = tt_ / (n_sat*n_cust)

        for i in N_C:
            S[i] = int(round(0.3*n_cust*tt_*q[i]/(0.7*total_q)))

        tt_ += sum(S.values()) / n_cust

        for i in N_C:
            t_avg = sum(t[s, i] for s in N_S) / n_sat
            c_center = random.randint(int(round(t_avg, 0)), int(round(4*tt_ - t_avg - S[i], 0)))
            w_width = random.gauss(S[i], 0.15*S[i])

            e[i] = max(int(round(c_center - w_width, 0)), 0)
            l[i] = int(round(c_center + w_width, 0))
            p[i] = 1

        total_S = sum(S.values())
        total_eL = 0
        total_Vs = 0

        for v in range(len(hv)):
            total_eL += F[v][5]*F[v][1]
            total_Vs += F[v][1]

        eta = max(1, round(5*total_S*total_Vs / (n_cust*total_eL)))
        T0 = 0

        # Write file
        instance_name = f"H{int(vtype)}-{name}"
        folder_name = f"./e-2e-vrp instances/H{int(vtype)}/{folder}"

        with open(f"{folder_name}/{instance_name}.txt", "w", encoding="utf-8") as f:
            f.write("!----------------------------------------------------------------\n")
        
            f.write("!Stores: (first: depot x,y; then: satellites x,y)\n")
            sat_strs = []
            for i in [N_d]+N_S:
                sat_strs.append(f"{int(coord[i][0])},{int(coord[i][1])}")
            f.write("   ".join(sat_strs) + "\n")
            f.write("!----------------------------------------------------------------\n")
        
            f.write("!Trucks: (total#, cap, cost/dist, fixcost)\n")
            trucks = []
            for v in T:
                trucks.append(",".join(str(x) for x in v))
            f.write("   ".join(trucks) + "\n")
            f.write("!----------------------------------------------------------------\n")
            
            f.write("!CityFreighters: (max cf/sat, total#, cap, cost/dist, fixcost, maxCharge, energyConsumption)\n")
            freighters = []
            for v in F:
                freighters.append(",".join(str(x) for x in v))
            f.write("   ".join(freighters) + "\n")
            f.write("!----------------------------------------------------------------\n")

            f.write("!Customers: (x, y, demand, readyTime, dueDate, serviceTime, penalizationCost)\n")
            customers = []
            for i in N_C:
                customers.append(f"{int(coord[i][0])},{int(coord[i][1])},{int(q[i])},{int(e[i])},{int(l[i])},{int(S[i])},{p[i]}")
            f.write("   ".join(customers) + "\n")
            f.write("!----------------------------------------------------------------\n")

            f.write("!Recharge stations: x,y\n")
            rst_strs = []
            for i in N_R:
                rst_strs.append(f"{int(coord[i][0])},{int(coord[i][1])}")
            f.write("   ".join(rst_strs) + "\n")

            f.write("\n")

            f.write(f"g inverse recharge rate {round(eta,3)}\n")
            f.write(f"T min departure time {round(T0,1)}\n")

    return instance_name, len(N_C), len(N_S), len(N_R)

In [14]:
import os

root_dir = r"./E2EVRP_instances"
instances = []

for root, dirs, files in os.walk(root_dir):
    for file in files:
        file_name = file.removesuffix(".dat")
        instance_root = os.path.join(root, file)
        folder = os.path.relpath(root, root_dir)

        if folder != ".":
            print(folder, file_name)

            for nveh in [1,2,3]:
                instance_name, N_C, N_S, N_R = get_instance(instance_root, nveh, folder, file_name)
                instances.append([folder, instance_name, N_C, N_S, N_R, nveh])

df = pd.DataFrame(instances, columns=["Set", "Instance name", "Customers", "Satellites", "Stations", "Fleet"])
df.to_excel("Instances.xlsx", sheet_name="Hoja1", index=False)

Set2 E-Set2a_E-n22-k4-s10-14_int
Set2 E-Set2a_E-n22-k4-s11-12_int
Set2 E-Set2a_E-n22-k4-s12-16_int
Set2 E-Set2a_E-n22-k4-s6-17_int
Set2 E-Set2a_E-n22-k4-s8-14_int
Set2 E-Set2a_E-n22-k4-s9-19_int
Set2 E-Set2a_E-n33-k4-s1-9_int
Set2 E-Set2a_E-n33-k4-s14-22_int
Set2 E-Set2a_E-n33-k4-s2-13_int
Set2 E-Set2a_E-n33-k4-s3-17_int
Set2 E-Set2a_E-n33-k4-s4-5_int
Set2 E-Set2a_E-n33-k4-s7-25_int
Set2 E-Set2b_E-n51-k5-s11-19-27-47_int
Set2 E-Set2b_E-n51-k5-s11-19_int
Set2 E-Set2b_E-n51-k5-s2-17_int
Set2 E-Set2b_E-n51-k5-s2-4-17-46_int
Set2 E-Set2b_E-n51-k5-s27-47_int
Set2 E-Set2b_E-n51-k5-s32-37_int
Set2 E-Set2b_E-n51-k5-s4-46_int
Set2 E-Set2b_E-n51-k5-s6-12-32-37_int
Set2 E-Set2b_E-n51-k5-s6-12_int
Set2 E-Set2c_E-n51-k5-s11-19-27-47_int
Set2 E-Set2c_E-n51-k5-s11-19_int
Set2 E-Set2c_E-n51-k5-s2-17_int
Set2 E-Set2c_E-n51-k5-s2-4-17-46_int
Set2 E-Set2c_E-n51-k5-s27-47_int
Set2 E-Set2c_E-n51-k5-s32-37_int
Set2 E-Set2c_E-n51-k5-s4-46_int
Set2 E-Set2c_E-n51-k5-s6-12-32-37_int
Set2 E-Set2c_E-n51-k5-s6-12_

In [6]:
from __future__ import annotations

import math
import random
import pandas as pd
from pathlib import Path
from openpyxl import load_workbook

# -------------------------
def _parse_groups(line):
    G = [g for g in line.strip().split() if g]
    l_groups = []
    for g in G:
        l_groups.append([float(x) for x in g.split(",")])
    return l_groups

def _fmt_groups(groups):
    return "   ".join(",".join(str(int(round(v))) for v in g) for g in groups)

def _euclid(p, q):
    return math.hypot(p[0] - q[0], p[1] - q[1])

def _pick_nearest(points, k, rng: random.Random):
    if k <= 0:
        return []
    if k >= len(points):
        return points[:]

    seed_pt = rng.choice(points)
    sxy = (seed_pt[0], seed_pt[1])
    ordered = sorted(points, key=lambda row: _euclid((row[0], row[1]), sxy))
    return ordered[:k]

def _scale_count(old, ratio):
    if old <= 0:
        return 0
    return min(max(1, int(round(old*ratio))) + 2, old)

# -------------------------
def _read_instance_txt(path):
    lines = path.read_text(encoding="utf-8", errors="ignore").splitlines()

    def find_line_containing(substr: str) -> int:
        for i, ln in enumerate(lines):
            if ln.strip().startswith(substr):
                return i
        raise ValueError(f"No encontré la sección: {substr}")

    idx_stores = find_line_containing("!Stores:")
    idx_trucks = find_line_containing("!Trucks:")
    idx_cfs = find_line_containing("!CityFreighters:")
    idx_customers = find_line_containing("!Customers:")
    idx_rs = find_line_containing("!Recharge stations:")

    # Las líneas de datos están justo después del encabezado
    stores_line = lines[idx_stores + 1].strip()
    trucks_line = lines[idx_trucks + 1].strip()
    cfs_line = lines[idx_cfs + 1].strip()
    customers_line = lines[idx_customers + 1].strip()
    rs_line = lines[idx_rs + 1].strip()

    # Parámetros al final (g y T) en tu ejemplo
    # buscamos líneas que empiecen con 'g ' y 'T '
    g_line = next((ln.strip() for ln in lines if ln.strip().startswith("g ")), None)
    T_line = next((ln.strip() for ln in lines if ln.strip().startswith("T ")), None)

    if g_line is None or T_line is None:
        raise ValueError("No encontré líneas de parámetros 'g ...' y/o 'T ...'.")

    return {"raw_lines": lines,
            "stores_line": stores_line,
            "trucks_line": trucks_line,
            "cfs_line": cfs_line,
            "customers_line": customers_line,
            "rs_line": rs_line,
            "g_line": g_line,
            "T_line": T_line}

def make_reduced_instance(input_txt, n_satellites, n_recharge_stations, n_customers, name):
    input_txt = Path(input_txt)

    data = _read_instance_txt(input_txt)
    rng = random.Random(42)

    # ---- Stores: (depot + satellites)
    stores = _parse_groups(data["stores_line"])
    depot = stores[0]
    satellites = stores[1:]
    satellites_new = _pick_nearest(satellites, n_satellites, rng)
    stores_new = [depot] + satellites_new

    # ---- Recharge stations
    rs = _parse_groups(data["rs_line"])
    rs_new = _pick_nearest(rs, n_recharge_stations, rng)

    # ---- Customers (7 fields)
    customers = _parse_groups(data["customers_line"])
    old_ncust = len(customers)
    customers_new = _pick_nearest(customers, n_customers, rng)

    # ---- Pseudo-lineal fleet scaling
    ratio = (len(customers_new) / old_ncust) if old_ncust > 0 else 1.0

    # Trucks: grupos de 4: (total#, cap, cost/dist, fixcost)
    trucks = _parse_groups(data["trucks_line"])
    n_trucks = len(trucks)
    for g in trucks:
        g[0] = float(_scale_count(int(round(g[0])), ratio))
    trucks_new = trucks

    # CityFreighters: grupos de 7: (max cf/sat, total#, cap, cost/dist, fixcost, maxCharge, energyConsumption)
    cfs = _parse_groups(data["cfs_line"])
    for g in cfs:
        # dejamos max cf/sat igual (g[0]) y escalamos total# (g[1])
        g[1] = float(_scale_count(int(round(g[1])), ratio))
    cfs_new = cfs

    # ---- Construir salida (mismo “estilo” que tu archivo)
    out_lines = []
    out_lines += [
        "!----------------------------------------------------------------",
        "!Stores: (first: depot x,y; then: satellites x,y)",
        _fmt_groups(stores_new),
        "!----------------------------------------------------------------",
        "!Trucks: (total#, cap, cost/dist, fixcost)",
        _fmt_groups(trucks_new),
        "!----------------------------------------------------------------",
        "!CityFreighters: (max cf/sat, total#, cap, cost/dist, fixcost, maxCharge, energyConsumption)",
        _fmt_groups(cfs_new),
        "!----------------------------------------------------------------",
        "!Customers: (x, y, demand, readyTime, dueDate, serviceTime, penalizationCost)",
        _fmt_groups(customers_new),
        "!----------------------------------------------------------------",
        "!Recharge stations: x,y",
        _fmt_groups(rs_new),
        "",
        data["g_line"],
        data["T_line"],
        "",
    ]
    
    # Write file
    folder_name = f"./e-2e-vrp instances/H{int(n_trucks)}/Small"

    with open(f"{folder_name}/{name}.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(out_lines))


#############################
workbook = load_workbook("./Instances.xlsx")
sheet = workbook['Hoja1']
count = 1
instances = []

for i in range(1461):
    folder = sheet.cell(2+i, 1).value
    instance_name = sheet.cell(2+i, 2).value
    nC = sheet.cell(2+i, 3).value
    nS = sheet.cell(2+i, 4).value
    nR = sheet.cell(2+i, 5).value
    fleet = sheet.cell(2+i, 6).value

    if nC is not None and nC < 50:
        input_txt = f"./e-2e-vrp instances/H{fleet}/{folder}/{instance_name}.txt"

        for n in [5,10,15]:
            ratio = n/10
            n_sat = min(max(1, round(n/10)), nS)
            n_rec = min(max(1, round(n/5))+ 1, nR)
            name = f"I{count}-{n}-{n_sat}-{n_rec}-H{fleet}"
            make_reduced_instance(input_txt, n_sat, n_rec, n, name)
            instances.append(["Small", name, n, n_sat, n_rec, fleet])
        count += 1

df = pd.DataFrame(instances, columns=["Set", "Instance name", "Customers", "Satellites", "Stations", "Fleet"])
df.to_excel("Instances_small.xlsx", sheet_name="Hoja1", index=False)